# Create COSTECH NFAST Awards

Creates OpenAlex award rows from the Tanzania Commission for Science and Technology (COSTECH) public NFAST funded-projects API.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/costech_to_s3.py` to fetch the official NFAST API, write parquet, and upload it.
- Contractor has prepared the script, notebook, registry entry, tracker row, and local QA, but does not have S3/Databricks access.
- The downloader writes parquet columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data source:** `https://www.costech.or.tz/projects/costech-funded` calls `https://rclearance.costech.or.tz/api/v1/nfast/project/list`  
**S3 location:** `s3a://openalex-ingest/awards/costech/costech_nfast_projects.parquet`  
**Local full-source validation on 2026-05-27:** 421 official COSTECH NFAST project rows, 2005-2025, 421 unique native IDs, 100% title/PI/program/subprogram/sector/status/source-funder/budget/date/landing-url coverage, 98.3% description coverage, 99.8% positive TZS budgets, and TZS 38,206,439,359 total budget.

**OpenAlex funder mapping:** Tanzania Commission for Science and Technology `F4320323478` (TZ, ROR `https://ror.org/03e04g978`, DOI `10.13039/501100005883`).

**Mapping summary:** One OpenAlex award row per NFAST project. The API has a source `funder` label (`Government of Tanzania`, `SIDA`, `HDIF`, `NORAD`, `IDRC`) that is preserved in the raw table for audit; the OpenAlex funder is COSTECH because the records are published by COSTECH through its official funded-projects portal and NFAST API. `total_budget` maps to `amount` in TZS; program/subprogram map to `funder_scheme`; researcher names map to `lead_investigator`.


## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.costech_awards_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/costech/costech_nfast_projects.parquet`;

In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-27 found 421 rows.
SELECT COUNT(*) as total_costech_projects
FROM openalex.awards.costech_awards_raw;

In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.costech_awards_raw;

In [ ]:
%sql
-- Sample raw COSTECH data.
SELECT
    funder_award_id,
    display_name,
    lead_investigator_name,
    program_name,
    subprogram_name,
    sector_name,
    source_funder_name,
    amount,
    currency,
    start_date,
    end_date,
    landing_page_url
FROM openalex.awards.costech_awards_raw
LIMIT 10;

## Step 1.5: Inspect Raw Data, Money, Dates, and Native Keys

In [ ]:
%sql
-- Money-field scan per runbook Step 1.5.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'costech_awards_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|grant|award|tzs';

In [ ]:
%sql
-- Confirm amount/date coverage before mapping.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_with_amount,
    SUM(CASE WHEN TRY_CAST(amount AS DOUBLE) > 0 THEN 1 ELSE 0 END) AS rows_with_positive_amount,
    ROUND(try_divide(SUM(CASE WHEN TRY_CAST(amount AS DOUBLE) > 0 THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_positive_amount,
    ROUND(MIN(TRY_CAST(amount AS DOUBLE)), 0) AS min_amount,
    ROUND(MAX(TRY_CAST(amount AS DOUBLE)), 0) AS max_amount,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_budget_tzs,
    COUNT(start_date) AS rows_with_start_date,
    COUNT(end_date) AS rows_with_end_date,
    MIN(TRY_CAST(source_year AS INT)) AS min_year,
    MAX(TRY_CAST(source_year AS INT)) AS max_year
FROM openalex.awards.costech_awards_raw;

In [ ]:
%sql
-- Native-key inspection: distinct IDs must equal total rows.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT source_project_id) AS distinct_source_project_ids,
    COUNT(DISTINCT landing_page_url) AS distinct_landing_pages
FROM openalex.awards.costech_awards_raw;

In [ ]:
%sql
-- Program/subprogram distribution.
SELECT
    program_name,
    subprogram_name,
    COUNT(*) AS cnt,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_tzs
FROM openalex.awards.costech_awards_raw
GROUP BY program_name, subprogram_name
ORDER BY cnt DESC, program_name, subprogram_name;

In [ ]:
%sql
-- Source funding labels are preserved for audit; OpenAlex funder remains COSTECH.
SELECT
    source_funder_name,
    COUNT(*) AS cnt,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_tzs
FROM openalex.awards.costech_awards_raw
GROUP BY source_funder_name
ORDER BY cnt DESC;

In [ ]:
%sql
-- Year, sector, and description coverage.
SELECT
    COUNT(*) AS total,
    COUNT(description) AS has_description,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description,
    COUNT(sector_name) AS has_sector,
    MIN(TRY_CAST(source_year AS INT)) AS min_year,
    MAX(TRY_CAST(source_year AS INT)) AS max_year
FROM openalex.awards.costech_awards_raw;

## Step 1.6: Funder Existence Fail-Fast

In [ ]:
%sql
-- Mandatory runbook assertion: fail if the funder is missing from openalex.common.funder.
SELECT assert_true(
    COUNT(*) = 1,
    'COSTECH funder F4320323478 is missing from openalex.common.funder; stop before transforming awards.'
) AS costech_funder_exists
FROM openalex.common.funder
WHERE funder_id = 4320323478;

In [ ]:
%sql
-- Confirm canonical funder fields used by the transform.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320323478;

## Step 2: Transform to OpenAlex Award Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.costech_awards
USING delta
AS
WITH
costech_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320323478
),

raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        CASE WHEN TRY_CAST(amount AS DOUBLE) IS NOT NULL THEN 'TZS' ELSE NULL END AS parsed_currency,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS parsed_start_date,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS parsed_end_date,
        TRY_CAST(source_year AS INT) AS parsed_year
    FROM openalex.awards.costech_awards_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 as id,

        TRIM(g.display_name) as display_name,

        CASE
            WHEN g.description IS NULL OR TRIM(g.description) = '' THEN NULL
            ELSE TRIM(g.description)
        END as description,

        f.funder_id,
        g.native_award_id as funder_award_id,

        g.parsed_amount as amount,
        g.parsed_currency as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        COALESCE(NULLIF(TRIM(g.funding_type), ''), 'research') as funding_type,

        COALESCE(NULLIF(TRIM(g.funder_scheme), ''), NULLIF(TRIM(g.subprogram_name), ''), NULLIF(TRIM(g.program_name), ''), 'NFAST projects') as funder_scheme,

        'costech_nfast_projects' as provenance,

        g.parsed_start_date as start_date,
        g.parsed_end_date as end_date,
        COALESCE(YEAR(g.parsed_start_date), g.parsed_year) as start_year,
        COALESCE(YEAR(g.parsed_end_date), g.parsed_year) as end_year,

        struct(
            NULLIF(TRIM(g.lead_investigator_given_name), '') as given_name,
            NULLIF(TRIM(g.lead_investigator_family_name), '') as family_name,
            CAST(NULL AS STRING) as orcid,
            g.parsed_start_date as role_start,
            struct(
                CAST(NULL AS STRING) as name,
                'TZ' as country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
            ) as affiliation
        ) as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        NULLIF(TRIM(g.landing_page_url), '') as landing_page_url,
        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM raw_prepared g
    CROSS JOIN costech_funder f
)

SELECT * FROM awards_transformed;

## Step 3: Delete Previous Source Rows and Insert Priority 152

In [ ]:
%sql
-- Remove previous COSTECH data before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'costech_nfast_projects' AND priority = 152;

-- Insert into openalex_awards_raw with exact OpenAlex awards schema plus priority.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    152 as priority
FROM openalex.awards.costech_awards;

## Step 6: Verification and Admin QA

In [ ]:
%sql
SELECT COUNT(*) as total_costech_awards
FROM openalex.awards.costech_awards;

In [ ]:
%sql
-- Confirm the transformed table has the OpenAlex awards columns only.
DESCRIBE openalex.awards.costech_awards;

In [ ]:
%sql
-- Sample transformed awards.
SELECT
    id,
    display_name,
    funder_award_id,
    funder_scheme,
    funding_type,
    amount,
    currency,
    start_date,
    end_date,
    lead_investigator.given_name AS pi_given,
    lead_investigator.family_name AS pi_family,
    landing_page_url
FROM openalex.awards.costech_awards
LIMIT 10;

In [ ]:
%sql
-- Check ID and native-key uniqueness.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS distinct_openalex_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.costech_awards;

In [ ]:
%sql
-- Check funder distribution. Should be only F4320323478.
SELECT funder.display_name, funder_id, COUNT(*) as cnt
FROM openalex.awards.costech_awards
GROUP BY funder.display_name, funder_id
ORDER BY cnt DESC;

In [ ]:
%sql
-- Data completeness.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(landing_page_url) as has_url,
    COUNT(lead_investigator.family_name) as has_pi_family,
    SUM(amount) as total_funding,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_with_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) as pct_description,
    ROUND(try_divide(COUNT(lead_investigator.family_name), COUNT(*)) * 100.0, 1) as pct_pi_family
FROM openalex.awards.costech_awards;

In [ ]:
%sql
-- Amount/currency fail-fast check for this grant pattern.
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS rows_with_positive_amount,
    ROUND(try_divide(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_positive_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    ROUND(AVG(CASE WHEN amount > 0 THEN amount END), 0) AS avg_positive_amount,
    ROUND(SUM(amount), 0) AS total_amount
FROM openalex.awards.costech_awards;

In [ ]:
%sql
-- Year distribution.
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_tzs
FROM openalex.awards.costech_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year DESC;

In [ ]:
%sql
-- Funder scheme distribution.
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_tzs
FROM openalex.awards.costech_awards
GROUP BY funder_scheme
ORDER BY cnt DESC;

In [ ]:
%sql
-- Verify rows inserted into the raw awards table at priority 152.
SELECT
    priority,
    provenance,
    COUNT(*) as cnt,
    COUNT(DISTINCT funder_award_id) as distinct_funder_award_ids,
    ROUND(SUM(amount), 0) as total_tzs
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'costech_nfast_projects' AND priority = 152
GROUP BY priority, provenance;

## Admin Handoff Notes

- Contractor validation stopped at local parquet generation and notebook preparation.
- Admin must upload `costech_nfast_projects.parquet` to `s3a://openalex-ingest/awards/costech/costech_nfast_projects.parquet`.
- Admin must run this notebook on Databricks, inspect Step 6 counts/coverage, then run the combined `CreateAwards.ipynb` after QA.
- Do not add job YAML until admin has run and verified the ingest.
